<a href="https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/%20w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one page (`content_hash_id`), on one day (`report_date`)**, in
`fact_content_daily_performance` — verified below with a grain query.

**Table(s):** `fact_content_daily_performance` (daily metrics) joined to
`dim_content` (static page metadata) on `content_hash_id`; `dim_clients`
only for `gsc_data_start`/`ga4_data_start` context, not as a feature source.

**Time window:** mid-panel month `month = '2026-03'`, split into a first half (`report_date < '2026-03-16'`, my feature window) and second half (`report_date >= '2026-03-16'`, my label window) — per the warning never to develop label logic on the sealed final month.

## 2. Fields: Feature / Label / Context / Excluded

**Feature (first-half window, knowable before the decision):**
- `gsc_impressions`, `gsc_clicks` (summed, first half)
- `gsc_avg_position` (averaged, first half)
- `content_created_date`, `content_updated_date` (from `dim_content`, static)

**Label (second-half window, the future outcome):**
- `is_declining` = 1 if second-half `gsc_clicks` < first-half `gsc_clicks`, else 0

**Context (joins/grouping only, never a feature):**
- `content_hash_id`, `client_hash_id` — used for the client-grouped split

**Excluded, deliberately:**
- `ga4_*` and `sessions_*` columns for this pass — `gsc_data_available` and
  `ga4_data_available` can differ per row (unbalanced panel), and mixing an
  incomplete GA4 signal into features this early would add noise I can't
  yet account for. I'm keeping this contract to GSC-only signals until I
  confirm GA4 availability separately.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"
DIM  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# Query 1 — grain: one row really is one page-day

grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{FACT}', hive_partitioning=1)
    WHERE month = '2026-03'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print("Rows violating one-row-per-page-per-day grain:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating one-row-per-page-per-day grain: 0


In [ ]:
# Query 2 — row count and date span for the slice

span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{FACT}', hive_partitioning=1)
    WHERE month = '2026-03'
""").df()
print(span_check)

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


In [ ]:
# Query 3 — availability, filtered with IS TRUE

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS available_rows
    FROM read_parquet('{FACT}', hive_partitioning=1)
    WHERE month = '2026-03'
""").df()
print(availability_check)

   total_rows  available_rows
0     9841378       3611061.0


In [ ]:
# Five Features + The Trap

feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        d.content_created_date,
        d.content_updated_date,
        DATE '2026-03-16' - d.content_created_date AS content_age_days,
        DATE '2026-03-16' - d.content_updated_date AS days_since_last_update,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_first_half,
        AVG(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_avg_position END) AS avg_position_first_half,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_second_half
    FROM read_parquet('{FACT}', hive_partitioning=1) f
    JOIN read_parquet('{DIM}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, f.client_hash_id, d.content_created_date, d.content_updated_date
""").df()

feature_frame['is_declining'] = (feature_frame.clicks_second_half < feature_frame.clicks_first_half).astype(int)
print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 11)


,content_hash_id,client_hash_id,content_created_date,content_updated_date,content_age_days,days_since_last_update,impressions_first_half,clicks_first_half,avg_position_first_half,clicks_second_half,is_declining
0,content_14a3d47ccd0d15dc,client_2094c6eb080311d5,2025-12-09,2026-05-12,97,-57,2.0,0.0,5.500000,0.0,0
1,content_14a6f92117604fef,client_2094c6eb080311d5,2025-12-11,2026-05-20,95,-65,0.0,0.0,NaN,0.0,0
2,content_1515e6f85acc54e1,client_2094c6eb080311d5,2026-02-10,2026-06-01,34,-77,10.0,0.0,17.208333,0.0,0
3,content_153357ef5824c7cc,client_2094c6eb080311d5,2026-03-03,2026-05-20,13,-65,8.0,0.0,29.625000,0.0,0
4,content_15770c63daac443b,client_2094c6eb080311d5,2026-02-04,2026-06-29,40,-105,765.0,1.0,5.870152,3.0,0


- **content_age_days**: knowable because `content_created_date` is a static fact recorded when the page was made, always known before any decision.
- **days_since_last_update**: same — a static content-ops fact.
- **impressions_first_half**: knowable because it's summed only over dates strictly before the decision point (`report_date < 2026-03-16`).
- **clicks_first_half**: same reasoning — a prior-window signal only.
- **avg_position_first_half**: averaged only over the first-half window, so it reflects ranking behavior known before the decision moment.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

honest_features = ['content_age_days', 'days_since_last_update',
                    'impressions_first_half', 'clicks_first_half', 'avg_position_first_half']
X_honest = feature_frame[honest_features].fillna(0)
y = feature_frame['is_declining']

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X_honest)[:, 1])
print(f"Honest AUC: {honest_auc:.3f}")

# THE TRAP — add the label-derived column on purpose
feature_frame['leaky_feature'] = feature_frame['clicks_second_half']
X_leaky = feature_frame[honest_features + ['leaky_feature']].fillna(0)
leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:, 1])
print(f"Leaky AUC (with clicks_second_half as a feature): {leaky_auc:.3f}  <- jumps toward 1.0")

# delete the trap, keep the honest number
print(f"Final honest AUC kept: {honest_auc:.3f}")

Honest AUC: 0.938
Leaky AUC (with clicks_second_half as a feature): 0.974  <- jumps toward 1.0
Final honest AUC kept: 0.938


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** `gsc_data_available` varies row by row — this is an unbalanced panel where per-client GSC history depth differs
(`dim_clients.gsc_data_start`). Filtering to `gsc_data_available IS TRUE`
drops rows unevenly across clients, so my March 2026 slice may over-represent clients with longer-established GSC integration and under-represent newer ones — a pattern found here may not generalize evenly across all clients.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.